# Daily Challenge — Build a Tiny Agent with Tools

## Complete teacher solution, thoroughly commented

This notebook creates a small `smolagents` tool-calling agent that can:

- add two numbers;
- multiply two numbers;
- search an internal knowledge base;
- preserve source citations;
- expose tool calls and observations;
- respond honestly when evidence is missing.

A deterministic `smolagents.Model` is used by default, so no API key and no
model download are required.

## Agent workflow

```text
User request
    ↓
Model chooses one typed tool call
    ↓
Trusted Python tool executes
    ↓
Observation is recorded in agent memory
    ↓
Model calls final_answer
```

# 0. Install and import

In [ ]:
# Version 1.26.0 is pinned because smolagents is an experimental library
# whose interfaces may change between releases.

%pip install -qU \
    "smolagents[transformers]==1.26.0" \
    "wikipedia>=1.4,<2"

In [ ]:
import importlib.metadata as metadata
import json
import re
from typing import Any

from smolagents import (
    Model,
    Tool,
    ToolCallingAgent,
    TransformersModel,
)

from smolagents.models import (
    ChatMessage,
    ChatMessageToolCall,
    ChatMessageToolCallFunction,
    MessageRole,
)

print("smolagents:", metadata.version("smolagents"))

# 1. Define a small source-tagged knowledge base

In [ ]:
kb_snippets = [
    {
        "source": "kb:1",
        "text": (
            "An agentic AI loop observes a task, plans the next action, "
            "selects a tool when needed, inspects the observation, and "
            "continues until it can answer."
        ),
        "keywords": [
            "agentic",
            "agent",
            "loop",
            "plan",
            "planning",
            "observation",
        ],
    },
    {
        "source": "kb:2",
        "text": (
            "Tools give an agent controlled capabilities such as "
            "calculation, search, retrieval, or database access."
        ),
        "keywords": [
            "tool",
            "tools",
            "capability",
            "calculation",
            "search",
        ],
    },
    {
        "source": "kb:3",
        "text": (
            "A tool-calling agent asks a model for a structured tool name "
            "and arguments, then trusted application code executes them."
        ),
        "keywords": [
            "tool-calling",
            "structured",
            "arguments",
            "execute",
            "json",
        ],
    },
    {
        "source": "kb:4",
        "text": (
            "A grounded answer cites the evidence used for its claims and "
            "does not invent a source identifier."
        ),
        "keywords": [
            "grounded",
            "citation",
            "cite",
            "evidence",
            "source",
        ],
    },
    {
        "source": "kb:5",
        "text": (
            "When evidence is missing, the agent should say that it lacks "
            "support and propose a more specific follow-up question."
        ),
        "keywords": [
            "missing",
            "unknown",
            "evidence",
            "follow-up",
            "followup",
        ],
    },
    {
        "source": "kb:6",
        "text": (
            "Short answers are easier to verify: give the main conclusion "
            "and one supporting statement in two to four sentences."
        ),
        "keywords": [
            "short",
            "concise",
            "answer",
            "verify",
            "sentences",
        ],
    },
    {
        "source": "kb:7",
        "text": (
            "An agent should avoid unnecessary tool calls and should not "
            "repeat an identical action after receiving its observation."
        ),
        "keywords": [
            "repeat",
            "unnecessary",
            "action",
            "observation",
            "efficient",
        ],
    },
]

print("KB entries:", len(kb_snippets))

for item in kb_snippets:
    print(f"[{item['source']}] {item['text']}")

# 2. Implement the two tools

In [ ]:
# Ignore common words during lexical matching.
KB_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can",
    "do", "does", "for", "from", "how", "i", "in", "is", "it",
    "of", "on", "or", "the", "this", "to", "what", "when",
    "where", "which", "who", "why", "with",
}


def tokenize(text: str) -> set[str]:
    """Convert text into normalized meaningful tokens."""

    tokens = re.findall(
        r"[a-z0-9]+(?:-[a-z0-9]+)?",
        text.lower(),
    )

    return {
        token
        for token in tokens
        if token not in KB_STOPWORDS
        and len(token) > 1
    }

In [ ]:
class KBLookupTool(Tool):
    """Search a tiny internal KB and preserve its source tags."""

    name = "kb_lookup"

    description = (
        "Search the internal knowledge base for evidence about agents, "
        "agentic loops, tools, grounding, citations, and missing evidence. "
        "Use this for conceptual questions in those areas."
    )

    inputs = {
        "query": {
            "type": "string",
            "description": (
                "The conceptual question or keywords to search for."
            ),
        },
    }

    output_type = "string"

    def __init__(
        self,
        kb: list[dict[str, Any]],
        max_results: int = 3,
    ):
        # Tool.__init__ validates the declared schema.
        super().__init__()

        self.kb = kb
        self.max_results = max_results

    def forward(self, query: str) -> str:
        """Return the best source-tagged matches."""

        query_terms = tokenize(query)

        if not query_terms:
            return (
                "No KB evidence found. Ask a more specific "
                "follow-up question."
            )

        ranked_matches = []

        for item in self.kb:
            searchable_terms = (
                tokenize(item["text"])
                | set(item.get("keywords", []))
            )

            # A larger overlap means the passage is more relevant.
            score = len(
                query_terms
                & searchable_terms
            )

            if score > 0:
                ranked_matches.append(
                    (score, item)
                )

        ranked_matches.sort(
            key=lambda pair: pair[0],
            reverse=True,
        )

        selected = [
            item
            for _score, item
            in ranked_matches[: self.max_results]
        ]

        if not selected:
            return (
                "No KB evidence found. Ask a more specific "
                "follow-up question."
            )

        return "\n".join(
            f"[{item['source']}] {item['text']}"
            for item in selected
        )

In [ ]:
class MathTool(Tool):
    """Add or multiply exactly two numbers."""

    name = "math_tool"

    description = (
        "Add or multiply exactly two numbers. "
        "Set op to either 'add' or 'multiply'."
    )

    inputs = {
        "a": {
            "type": "number",
            "description": "The first number.",
        },
        "b": {
            "type": "number",
            "description": "The second number.",
        },
        "op": {
            "type": "string",
            "description": (
                "The operation: 'add' or 'multiply'."
            ),
        },
    }

    output_type = "string"

    @staticmethod
    def _format_number(value: float) -> str:
        """Avoid displaying 42.0 when the result is a whole number."""

        if float(value).is_integer():
            return str(int(value))

        return str(value)

    def forward(
        self,
        a: float,
        b: float,
        op: str,
    ) -> str:
        """Validate and perform one arithmetic operation."""

        operation = op.strip().lower()

        if operation == "add":
            result = a + b
        elif operation == "multiply":
            result = a * b
        else:
            raise ValueError(
                "op must be 'add' or 'multiply'."
            )

        return self._format_number(result)

In [ ]:
kb_tool = KBLookupTool(
    kb=kb_snippets,
    max_results=3,
)
math_tool = MathTool()

# Direct tool tests isolate tool correctness from model behavior.
assert math_tool.forward(12, 30, "add") == "42"
assert math_tool.forward(7, 6, "multiply") == "42"
assert "[kb:1]" in kb_tool.forward(
    "What is an agentic AI loop?"
)

print("Direct tool checks passed.")

# 3. Implement a deterministic smolagents model

The stub below implements the official `Model.generate(...)` interface and
returns structured `ChatMessageToolCall` objects.

It provides a reliable assignment run while still exercising the real
`ToolCallingAgent` loop.

In [ ]:
def content_to_text(content: Any) -> str:
    """Flatten a message content value into plain text."""

    if content is None:
        return ""

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            str(block.get("text", ""))
            if isinstance(block, dict)
            else str(block)
            for block in content
        )

    return str(content)


def extract_original_task(
    messages: list[ChatMessage],
) -> str:
    """Read the latest user task from agent history."""

    for message in reversed(messages):
        if message.role == MessageRole.USER:
            return re.sub(
                r"^New task:\s*",
                "",
                content_to_text(
                    message.content
                ).strip(),
                flags=re.IGNORECASE,
            )

    return ""


def extract_observation(
    messages: list[ChatMessage],
) -> str | None:
    """Read the latest tool response from agent history."""

    for message in reversed(messages):
        if message.role == MessageRole.TOOL_RESPONSE:
            return re.sub(
                r"^Observation:\s*",
                "",
                content_to_text(
                    message.content
                ).strip(),
                flags=re.IGNORECASE,
            )

    return None


def make_tool_call(
    name: str,
    arguments: dict[str, Any],
    call_id: str,
) -> ChatMessage:
    """Construct one structured assistant tool call."""

    return ChatMessage(
        role=MessageRole.ASSISTANT,
        content="",
        tool_calls=[
            ChatMessageToolCall(
                function=ChatMessageToolCallFunction(
                    name=name,
                    arguments=arguments,
                ),
                id=call_id,
                type="function",
            )
        ],
    )

In [ ]:
class DeterministicToolModel(Model):
    """No-token model that chooses tools with transparent rules."""

    def __init__(self):
        super().__init__(
            model_id="deterministic-tool-stub"
        )

    @staticmethod
    def _extract_numbers(
        task: str,
    ) -> list[float]:
        """Extract signed integers or decimal numbers in order."""

        return [
            float(value)
            for value in re.findall(
                r"-?\d+(?:\.\d+)?",
                task,
            )
        ]

    def generate(
        self,
        messages: list[ChatMessage],
        stop_sequences: list[str] | None = None,
        response_format: dict[str, str] | None = None,
        tools_to_call_from: list[Tool] | None = None,
        **kwargs,
    ) -> ChatMessage:
        """Choose a domain tool, then finish from its observation."""

        # The agent may pass these arguments to every Model.
        del stop_sequences, response_format, kwargs

        task = extract_original_task(
            messages
        )
        observation = extract_observation(
            messages
        )

        available_tools = {
            tool.name
            for tool in (
                tools_to_call_from or []
            )
        }

        # After a tool executes, convert the observation into a final answer.
        if observation is not None:
            if re.search(
                r"\b(add|sum|plus|multiply|times|product)\b",
                task.lower(),
            ):
                final_text = (
                    f"The result is {observation}."
                )
            elif observation.startswith(
                "No KB evidence found"
            ):
                final_text = (
                    "I do not have enough internal evidence to answer "
                    "reliably. Try a more specific follow-up question "
                    "that names the exact concept or workflow."
                )
            else:
                # Keep the first, highest-ranked KB passage and its citation.
                first_line = (
                    observation
                    .splitlines()[0]
                    .strip()
                )

                final_text = (
                    f"{first_line} "
                    "This answer is grounded in the cited internal source."
                )

            return make_tool_call(
                name="final_answer",
                arguments={
                    "answer": final_text,
                },
                call_id="stub-final-answer",
            )

        task_lower = task.lower()
        numbers = self._extract_numbers(
            task
        )

        multiply_requested = bool(
            re.search(
                r"\b(multiply|multiplied|times|product)\b",
                task_lower,
            )
        )

        add_requested = bool(
            re.search(
                r"\b(add|sum|plus)\b",
                task_lower,
            )
        )

        if (
            (multiply_requested or add_requested)
            and len(numbers) >= 2
            and "math_tool" in available_tools
        ):
            operation = (
                "multiply"
                if multiply_requested
                else "add"
            )

            return make_tool_call(
                name="math_tool",
                arguments={
                    "a": numbers[0],
                    "b": numbers[1],
                    "op": operation,
                },
                call_id="stub-math-call",
            )

        if "kb_lookup" in available_tools:
            return make_tool_call(
                name="kb_lookup",
                arguments={
                    "query": task,
                },
                call_id="stub-kb-call",
            )

        return make_tool_call(
            name="final_answer",
            arguments={
                "answer": (
                    "No suitable tool is available. "
                    "Please clarify the request."
                ),
            },
            call_id="stub-no-tool",
        )

# 4. Configure the default or optional local model

In [ ]:
# The reliable, free stub remains the default.
USE_LOCAL_TRANSFORMERS_MODEL = False

# This model is tiny enough for a demonstration, but it is not
# instruction-tuned for dependable structured tool calling.
LOCAL_MODEL_ID = "sshleifer/tiny-gpt2"

if USE_LOCAL_TRANSFORMERS_MODEL:
    model = TransformersModel(
        model_id=LOCAL_MODEL_ID,
        device_map="auto",
        max_new_tokens=256,
        do_sample=False,
    )

    print("Local model:", LOCAL_MODEL_ID)
else:
    model = DeterministicToolModel()

    print("Stub model:", model.model_id)

# 5. Instantiate the ToolCallingAgent

In [ ]:
agent = ToolCallingAgent(
    tools=[
        kb_tool,
        math_tool,
    ],
    model=model,

    # One domain tool call plus final_answer normally requires two steps.
    max_steps=3,

    # Level 2 displays the detailed action and observation flow.
    verbosity_level=2,

    instructions=(
        "Keep final answers between two and four short sentences. "
        "Use math_tool for arithmetic. "
        "Use kb_lookup for conceptual questions about agents and tools. "
        "When KB evidence is returned, preserve its [kb:N] citation. "
        "When evidence is missing, say so and suggest a specific "
        "follow-up question."
    ),
)

print(agent)

# 6. Inspect tool calls and observations

In [ ]:
def compact_agent_trace(
    tool_agent: ToolCallingAgent,
) -> list[dict[str, Any]]:
    """Extract a compact trace from the most recent agent run."""

    trace = []

    for step in tool_agent.memory.steps:
        tool_calls = getattr(
            step,
            "tool_calls",
            None,
        )

        if not tool_calls:
            continue

        for tool_call in tool_calls:
            trace.append({
                "tool": getattr(
                    tool_call,
                    "name",
                    "unknown",
                ),
                "arguments": getattr(
                    tool_call,
                    "arguments",
                    {},
                ),
                "observation": getattr(
                    step,
                    "observations",
                    None,
                ),
                "is_final_answer": getattr(
                    step,
                    "is_final_answer",
                    False,
                ),
            })

    return trace


def print_agent_trace(
    trace: list[dict[str, Any]],
) -> None:
    """Print one readable record per agent action."""

    for index, item in enumerate(
        trace,
        start=1,
    ):
        print(
            f"Step {index}: "
            f"tool={item['tool']}, "
            f"arguments={item['arguments']}, "
            f"observation={item['observation']!r}, "
            f"final={item['is_final_answer']}"
        )

# 7. Run the three required tests

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

run_records = []

for question in tests:
    print("\n" + "=" * 90)
    print("QUESTION:", question)

    # reset=True clears memory before every independent test.
    result = agent.run(
        question,
        reset=True,
    )

    trace = compact_agent_trace(
        agent
    )

    print("\nCOMPACT TOOL TRACE")
    print_agent_trace(trace)

    print("\nFINAL ANSWER")
    print(result)

    run_records.append({
        "question": question,
        "result": str(result),
        "trace": trace,
    })

In [ ]:
# Verify tool selection and final results.

assert "42" in run_records[0]["result"]
assert (
    run_records[0]["trace"][0]["tool"]
    == "math_tool"
)
assert (
    run_records[0]["trace"][0]["arguments"]["op"]
    == "add"
)

assert "42" in run_records[1]["result"]
assert (
    run_records[1]["trace"][0]["tool"]
    == "math_tool"
)
assert (
    run_records[1]["trace"][0]["arguments"]["op"]
    == "multiply"
)

assert "[kb:1]" in run_records[2]["result"]
assert (
    run_records[2]["trace"][0]["tool"]
    == "kb_lookup"
)

print("All required checks passed.")

# 8. Verify missing-evidence behavior

In [ ]:
missing_question = (
    "What is the maintenance schedule for the Orion spacecraft?"
)

missing_result = agent.run(
    missing_question,
    reset=True,
)

missing_trace = compact_agent_trace(
    agent
)

print_agent_trace(missing_trace)
print("\nFINAL ANSWER")
print(missing_result)

assert "do not have enough" in str(
    missing_result
).lower()

assert "follow-up" in str(
    missing_result
).lower()

print("Missing-evidence behavior passed.")

# Observations

- The agent, not the user, selects the tool.
- The tool schema tells the model which arguments are required.
- Trusted Python performs the actual calculation or retrieval.
- The agent memory records tool calls, arguments, observations, and the
  final answer.
- Source tags originate in the KB and are preserved in the answer.
- The deterministic stub is more reproducible than `tiny-gpt2` for
  structured tool calling.

# Deliverables checklist

- [x] Seven KB snippets
- [x] Source tags
- [x] `KBLookupTool`
- [x] keyword matching
- [x] no-evidence response
- [x] `MathTool`
- [x] addition
- [x] multiplication
- [x] deterministic `Model` stub
- [x] optional `TransformersModel`
- [x] `ToolCallingAgent`
- [x] three required questions
- [x] tool-call inspection
- [x] short final answers
- [x] `[kb:1]` citation
- [x] missing-evidence follow-up
- [x] commented and documented code

# References

- smolagents agents:
  https://huggingface.co/docs/smolagents/reference/agents
- smolagents tools:
  https://huggingface.co/docs/smolagents/reference/tools
- smolagents models:
  https://huggingface.co/docs/smolagents/reference/models